# Train the player detector on a Colab GPU

**Before running:** connect to the Colab kernel on a **GPU** runtime (Select Kernel → Colab → Auto Connect; runtime type = GPU / T4).

This version pulls the dataset **directly from Roboflow** — no 171 MB upload. In **cell 2**, paste your Roboflow API key (Roboflow → Settings → API, or the dataset page's "Download Dataset → show download code").

Run cells top to bottom:
1. confirm GPU  →  2. download from Roboflow  →  3. remap 10→5 classes  →  4. train  →  5. download `best.pt`.

When done, put `best.pt` into the project at `models/player_detector.pt` — the tracker auto-uses it (tracks the `player` class only, so refs/fans stop being boxed as players).

In [ ]:
# 1) Confirm we're on a GPU
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — set Runtime → Change runtime type → GPU")

In [ ]:
# 2) Pull the dataset straight from Roboflow (no upload needed).
#    Paste your API key below: Roboflow → Settings → API,  OR the dataset page's
#    "Download Dataset → show download code".
!pip install -q roboflow
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY")   # <-- replace with your key
ds = rf.workspace("roboflow-jvuqo").project("basketball-player-detection-3-ycjdo").version(18).download("yolov8")
ROOT = ds.location
print("dataset at:", ROOT)
import os
print("splits:", [d for d in ("train", "valid", "test") if os.path.isdir(f"{ROOT}/{d}")])

In [ ]:
# 3) Install ultralytics, remap 10 -> 5 classes in place, write data.yaml.
#    (Uses ROOT from cell 2 — Roboflow already extracted the dataset, no unzip needed.)
import subprocess, sys, glob, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

REMAP = {3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 1, 0: 2, 1: 2, 9: 3, 2: 4}
NEW_NAMES = ["player", "referee", "ball", "rim", "number"]

n = 0
for split in ("train", "valid", "test"):
    for f in glob.glob(f"{ROOT}/{split}/labels/*.txt"):
        out = []
        for line in open(f).read().splitlines():
            p = line.split()
            if p and int(p[0]) in REMAP:
                p[0] = str(REMAP[int(p[0])])
                out.append(" ".join(p))
        open(f, "w").write("\n".join(out) + ("\n" if out else ""))
        n += 1
print(f"remapped {n} label files -> {len(NEW_NAMES)} classes {NEW_NAMES}")

open(f"{ROOT}/data.yaml", "w").write(
    f"path: {ROOT}\ntrain: train/images\nval: valid/images\ntest: test/images\n"
    f"nc: {len(NEW_NAMES)}\nnames: {NEW_NAMES}\n")
print("wrote", f"{ROOT}/data.yaml")

In [ ]:
# 4) Train (yolov8m, 100 epochs, GPU). Watch mAP50 climb across the per-epoch table.
from ultralytics import YOLO
YOLO("yolov8m.pt").train(
    data=f"{ROOT}/data.yaml", epochs=100, imgsz=640, batch=16, device=0,
    project="player_runs", name="train", patience=20,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, fliplr=0.5, translate=0.1, scale=0.5, mosaic=1.0)
print("\nDONE -> download player_runs/train/weights/best.pt")

In [ ]:
# 5) Copy best.pt to your Google Drive (most reliable from Cursor's Colab extension,
#    where files.download() often doesn't trigger). Follow the auth prompt when it mounts.
import glob, shutil
from google.colab import drive
drive.mount('/content/drive')

hits = glob.glob("**/player_runs/train*/weights/best.pt", recursive=True)
print("found:", hits)
assert hits, "best.pt not found — check cell 4's 'Results saved to ...' path."
dst = "/content/drive/MyDrive/player_detector_best.pt"
shutil.copy(hits[0], dst)
print("\nCopied to your Google Drive →", dst)
print("Now get it onto your Mac: drive.google.com (or Drive desktop app) → download player_detector_best.pt")
print("Then move it to your project at  models/player_detector.pt")